In [1]:
import numpy as np
import pandas as pd

In [5]:
a = [1, 3,4]
b = [2, 3, 3]

In [6]:
a
b

[2, 3, 3]

## Способы создать numpy-массив (`ndarray`)

Все способы ниже возвращают объект `ndarray`. Главное отличие от обычного списка: у массива фиксированный тип элементов (`dtype`) и форма (`shape`), за счёт чего операции работают быстро и поэлементно.

In [ ]:
# 1. Из готовых данных — Python-списка или кортежа
print(np.array([1, 2, 3]))          # 1D-массив (вектор)
print(np.array(a))                  # из нашего списка a = [1, 3, 4]
print(np.array([[1, 2], [3, 4]]))   # 2D-массив (матрица) — список списков
print(np.asarray(b))                # asarray: как array, но НЕ копирует, если уже ndarray

In [ ]:
# 2. Заполненные одинаковым значением — форму (shape) задаём сами
print(np.zeros(5))         # [0. 0. 0. 0. 0.] — по умолчанию тип float
print(np.ones((2, 3)))     # матрица 2x3 из единиц (форма задаётся кортежем)
print(np.full((2, 2), 7))  # матрица 2x2, всё заполнено числом 7

In [ ]:
# 3. Числовые последовательности
print(np.arange(0, 10, 2))     # как range(): старт, стоп (не включая), шаг -> [0 2 4 6 8]
print(np.linspace(0, 1, 5))    # N точек равномерно от 0 до 1 ВКЛЮЧИТЕЛЬНО -> [0. 0.25 0.5 0.75 1.]

In [ ]:
# 4. Случайные числа (современный способ — через генератор)
rng = np.random.default_rng(42)      # 42 — seed: даёт воспроизводимый результат
print(rng.integers(0, 10, size=5))   # 5 целых из диапазона [0, 10)
print(rng.random(3))                 # 3 числа float из [0, 1)

In [ ]:
# 5. Спец-матрицы и явный тип данных (dtype)
print(np.eye(3))                          # единичная матрица 3x3 (1 на диагонали)
print(np.array([1, 2, 3], dtype=float))   # задать тип явно -> [1. 2. 3.]
print(np.zeros(3).dtype)                   # узнать тип элементов: float64

## `reshape` — изменить форму массива

`reshape` меняет **форму** (`shape`) массива, не меняя сами данные. Главное правило: **число элементов должно сохраняться** — например, 12 элементов можно сделать `(3, 4)`, `(4, 3)`, `(2, 6)`, `(2, 2, 3)`, но не `(3, 5)`.

In [ ]:
# Базовый пример: 12 чисел -> матрица 3x4
arr = np.arange(12)            # [0 1 2 ... 11], форма (12,)
print(arr.shape)              # (12,)

m = arr.reshape(3, 4)         # 3 строки, 4 столбца
print(m)
print(m.shape)               # (3, 4)

# Данные заполняются по строкам (слева направо, сверху вниз)
# Та же матрица, но 4x3 — порядок чисел сохраняется:
print(arr.reshape(4, 3))

In [ ]:
# -1 = "посчитай этот размер сам". Удобно, когда лень считать.
arr = np.arange(12)
print(arr.reshape(3, -1))   # фиксируем 3 строки -> столбцы numpy вычислит сам (4)
print(arr.reshape(-1, 2))   # фиксируем 2 столбца -> строк будет 6

# -1 можно использовать только в ОДНОМ измерении
print(arr.reshape(2, 2, -1).shape)   # (2, 2, 3) — трёхмерный массив

In [ ]:
# reshape возвращает ОБЫЧНО view (вид) на те же данные, а не копию!
base = np.arange(6)
view = base.reshape(2, 3)
view[0, 0] = 999        # меняем элемент во view...
print(base)             # ...и он меняется в исходном массиве: [999 1 2 3 4 5]

# Если нужна именно копия — добавь .copy()
safe = np.arange(6).reshape(2, 3).copy()

# Развернуть многомерный массив обратно в 1D:
print(view.ravel())     # ravel  -> по возможности view (быстро)
print(view.flatten())   # flatten -> всегда копия

In [ ]:
# Шаг 1. Загрузить одну страницу и посмотреть, что вернул pandas
import pandas as pd

URL = "https://almaty-marathon.kz/en/results/almaty_half_marathon_2026"

page1 = pd.read_html(URL + "?Results_page=1")[0]   # [0] — берём первую таблицу из списка
print("Колонки:", list(page1.columns))
print("Форма:  ", page1.shape)
page1.head()

In [ ]:
# Шаг 2. Соберём ВСЕ страницы в один DataFrame
def load_page(page):
    df = pd.read_html(f"{URL}?Results_page={page}")[0]
    df = df.dropna(subset=["Place"])        # убрать пустую строку-заголовок внутри таблицы
    df = df.drop(columns=["Certificate"])   # колонка-ссылка "Download" не нужна
    return df

# 186 страниц. Совет: сначала проверь на range(1, 6), потом ставь range(1, 187).
all_pages = [load_page(p) for p in range(1, 187)]
results = pd.concat(all_pages, ignore_index=True)   # склеить список DataFrame'ов в один
print("Всего строк:", len(results))
results.tail()

In [ ]:
# Шаг 3. Приведём типы данных, чтобы со столбцами можно было считать
# Внимание: в Place/Bib встречается 'DNF' (сошёл с дистанции).
# errors="coerce" превращает такие значения в NaN, а тип Int64 (с большой буквы)
# — это nullable-целое, которое умеет хранить пропуски.
results["Place"] = pd.to_numeric(results["Place"], errors="coerce").astype("Int64")
results["Bib"]   = pd.to_numeric(results["Bib"],   errors="coerce").astype("Int64")
results["Finish"]    = pd.to_timedelta(results["Finish"])     # "00:30:47" -> timedelta
results["Chip time"] = pd.to_timedelta(results["Chip time"])  # время по чипу

finishers = results["Place"].notna().sum()
print(results.dtypes, "\n")
print("Финишировали:    ", finishers, "из", len(results), "(остальные — DNF)")
print("Победитель:      ", results.loc[results["Place"].idxmin(), "Participant"])
print("Лучшее время:    ", results["Finish"].min())
print("Среднее время:   ", results["Finish"].mean())
results.head()

## Загрузка реальной таблицы: Almaty Half Marathon 2026

Источник: <https://almaty-marathon.kz/en/results/almaty_half_marathon_2026>

`pandas.read_html(url)` сам находит `<table>` на странице и возвращает **список** `DataFrame`. Результаты разбиты на 186 страниц по 30 строк — переключаются параметром `?Results_page=N`.

> Примечание: колонки `Citizenship` и `City` на сайте отрисованы картинками-флагами, поэтому в текст не извлекаются (будут пустыми).